# Treinamento YOLO Pose com K-Fold (Colab)

Este notebook assume que o dataset já está preparado em formato YOLO (images/labels) dentro de `dataset/fold_*`.

In [ ]:
!pip install -q ultralytics==8.3.193

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure os caminhos

Ajuste os caminhos abaixo para o seu Drive/ambiente Colab.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/content/drive/MyDrive/cow-classifier/dataset')
MODELS_DIR = Path('/content/drive/MyDrive/cow-classifier/models')
BASE_MODEL = 'yolo11x-pose.pt'
PROJECT_DIR = Path('/content/drive/MyDrive/cow-classifier/runs/kfold_pose')

EPOCHS = 100
IMGSZ = 640
BATCH = 8
DEVICE = 0  # 0 para GPU no Colab, ou 'cpu'
WORKERS = 2
RUN_PREFIX = 'train'
CONTINUE_FROM_BEST = False
PATIENCE = 50
SAVE_PERIOD = 10
CONF_MIN = 0.30

In [ ]:
import csv
import json
from datetime import datetime
from statistics import mean
from ultralytics import YOLO

def find_fold_yaml_files(dataset_root: Path):
    fold_yaml_files = []
    for fold_dir in sorted(dataset_root.glob('fold_*')):
        if not fold_dir.is_dir():
            continue
        yaml_candidates = sorted(fold_dir.glob('data_fold_*.yaml')) + sorted(fold_dir.glob('data_fold_*.yml'))
        if yaml_candidates:
            fold_yaml_files.append((fold_dir.name, yaml_candidates[0]))
    return fold_yaml_files

def read_metric(train_result, metric_key):
    results_dict = getattr(train_result, 'results_dict', None)
    if isinstance(results_dict, dict):
        return results_dict.get(metric_key)
    return None

def read_first_metric(train_result, keys):
    for key in keys:
        value = read_metric(train_result, key)
        if value is not None:
            return value
    return None

def existing_best_checkpoint(project_dir: Path, run_name: str):
    best_path = project_dir / run_name / 'weights' / 'best.pt'
    return best_path if best_path.exists() else None

def read_best_epoch_stats(results_csv_path: Path):
    if not results_csv_path.exists():
        return {}
    with results_csv_path.open('r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    pose_key = 'metrics/mAP50-95(P)'
    box_key = 'metrics/mAP50-95(B)'
    selected_key = pose_key if pose_key in rows[0] else box_key
    best_row = None
    best_value = float('-inf')
    for row in rows:
        raw = row.get(selected_key)
        if raw in (None, ''):
            continue
        value = float(raw)
        if value > best_value:
            best_value = value
            best_row = row
    if best_row is None:
        return {}
    return {
        'best_epoch': int(float(best_row.get('epoch', 0))),
        'metric_used': selected_key,
        'metric_value': best_value,
    }

def mean_or_none(values):
    valid = [v for v in values if isinstance(v, (int, float))]
    return mean(valid) if valid else None

def build_report(summary, conf_min):
    box_map50_values = [item.get('box_map50') for item in summary]
    box_map5095_values = [item.get('box_map50_95') for item in summary]
    pose_map50_values = [item.get('pose_map50') for item in summary]
    pose_map5095_values = [item.get('pose_map50_95') for item in summary]

    pose_best_fold = None
    best_pose_value = float('-inf')
    for item in summary:
        value = item.get('pose_map50_95')
        if isinstance(value, (int, float)) and value > best_pose_value:
            best_pose_value = value
            pose_best_fold = item

    return {
        'k_folds': len(summary),
        'aggregate_metrics': {
            'Box_mAP50_media_folds': mean_or_none(box_map50_values),
            'Box_mAP50_95_media_folds': mean_or_none(box_map5095_values),
            'Pose_mAP50_media_folds': mean_or_none(pose_map50_values),
            'Pose_mAP50_95_media_folds': mean_or_none(pose_map5095_values),
            'Pose_mAP50_95_melhor_fold': {
                'valor': None if pose_best_fold is None else pose_best_fold.get('pose_map50_95'),
                'fold': None if pose_best_fold is None else pose_best_fold.get('fold'),
            },
        },
        'metricas_teste_final': {
            'accuracy': None,
            'f1_macro': None,
            'top1_accuracy': None,
            'top3_accuracy': None,
            'top5_accuracy': None,
        },
        'metricas_com_rejeicao': {
            'confianca_min': conf_min,
            'cobertura': None,
            'accuracy_aceitas': None,
            'f1_macro_aceitas': None,
        },
        'folds': summary,
    }

def train_kfold(dataset_root, models_dir, base_model, project_dir, epochs, imgsz, batch, device, workers, run_prefix, continue_from_best, patience, save_period, conf_min):
    dataset_root = Path(dataset_root)
    models_dir = Path(models_dir)
    project_dir = Path(project_dir)
    base_model_path = models_dir / base_model

    if not dataset_root.exists():
        raise FileNotFoundError(f'Dataset root não encontrado: {dataset_root}')
    if not base_model_path.exists():
        raise FileNotFoundError(f'Modelo base não encontrado: {base_model_path}')

    fold_yaml_files = find_fold_yaml_files(dataset_root)
    if not fold_yaml_files:
        raise FileNotFoundError(f'Nenhum data_fold_*.yaml encontrado em {dataset_root}/fold_*')

    project_dir.mkdir(parents=True, exist_ok=True)
    print(f'Total de folds: {len(fold_yaml_files)}')

    summary = []
    for fold_name, data_yaml in fold_yaml_files:
        run_name = f'{run_prefix}_{fold_name}'
        model_start_path = base_model_path
        best_checkpoint = existing_best_checkpoint(project_dir, run_name)
        if continue_from_best and best_checkpoint is not None:
            model_start_path = best_checkpoint

        print(f'\n--- Iniciando {fold_name} ---')
        model = YOLO(str(model_start_path))
        result = model.train(
            data=str(data_yaml),
            task='pose',
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            workers=workers,
            device=device,
            project=str(project_dir),
            name=run_name,
            save=True,
            save_period=save_period,
            patience=patience,
            exist_ok=True,
        )

        save_dir = Path(getattr(result, 'save_dir', project_dir / run_name))
        summary.append({
            'fold': fold_name,
            'run': run_name,
            'box_map50': read_first_metric(result, ['metrics/mAP50(B)']),
            'box_map50_95': read_first_metric(result, ['metrics/mAP50-95(B)']),
            'pose_map50': read_first_metric(result, ['metrics/mAP50(P)']),
            'pose_map50_95': read_first_metric(result, ['metrics/mAP50-95(P)']),
            'best_weights': str(save_dir / 'weights' / 'best.pt'),
            'last_weights': str(save_dir / 'weights' / 'last.pt'),
            'best_epoch_stats': read_best_epoch_stats(save_dir / 'results.csv'),
        })

    report = build_report(summary, conf_min)
    report_path = project_dir / 'kfold_metrics_report.json'
    payload = {
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'settings': {
            'dataset_root': str(dataset_root),
            'base_model': str(base_model_path),
            'continue_from_best': continue_from_best,
            'epochs': epochs,
            'imgsz': imgsz,
            'batch': batch,
            'device': device,
            'workers': workers,
            'project': str(project_dir),
            'run_prefix': run_prefix,
            'save_period': save_period,
            'patience': patience,
        },
        'report': report,
    }
    with report_path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f'\nRelatório salvo em: {report_path}')
    return report_path

In [ ]:
report_path = train_kfold(
    dataset_root=DATASET_ROOT,
    models_dir=MODELS_DIR,
    base_model=BASE_MODEL,
    project_dir=PROJECT_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    run_prefix=RUN_PREFIX,
    continue_from_best=CONTINUE_FROM_BEST,
    patience=PATIENCE,
    save_period=SAVE_PERIOD,
    conf_min=CONF_MIN,
)
report_path